# Lista 5 — Zadanie 3: Eksploracja modeli encoder-only (20 pkt)

Porównujemy co najmniej **dwa aspekty**:
1. **Różne modele** klasyfikujące (HerBERT vs inny model z Hugging Face)
2. **Parametr `max_length`** — jak długi kontekst wpływa na wyniki (dobór na podstawie analizy długości tekstów)
3. **Temperatura** (`temperature`) — skalowanie logitów przed softmax (domyślnie 1.0)

Wyniki zestawiamy w tabeli porównawczej i analizujemy trudne przypadki.

In [ ]:
import sys

!{sys.executable} -m pip install -q torch transformers datasets scikit-learn pandas matplotlib

In [ ]:
import sys
from pathlib import Path

TASK5_DIR = Path("..").resolve()
if str(TASK5_DIR) not in sys.path:
    sys.path.insert(0, str(TASK5_DIR))

import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from common.data import load_polemo_test
from common.labels import map_text_to_class
from common.metrics import evaluate_predictions, print_evaluation

## Krok 1: Przygotowanie danych

In [ ]:
examples = load_polemo_test()
sentences = [ex["sentence"] for ex in examples]
y_true = [ex["class"] for ex in examples]

device = 0 if torch.cuda.is_available() else -1
print(f"Próbek: {len(sentences)} | Urządzenie: {'GPU' if device == 0 else 'CPU'}")

## Analiza długości tekstów

Przed doborem `max_length` sprawdzamy rozkład długości recenzji (w słowach), żeby opierać eksperyment na statystykach zbioru.

In [ ]:
# Analiza długości tekstów (w słowach)
lengths = [len(s.split()) for s in sentences]

print(f"Najkrótszy tekst (słowa): {min(lengths)}")
print(f"Najdłuższy tekst (słowa): {max(lengths)}")
print(f"Średnia długość: {sum(lengths) / len(lengths):.1f}")
print(f"Mediana: {pd.Series(lengths).median():.0f}")
print(f"90. percentyl: {pd.Series(lengths).quantile(0.9):.0f}")

plt.figure(figsize=(8, 4))
plt.hist(lengths, bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Liczba słów")
plt.ylabel("Liczba recenzji")
plt.title("Rozkład długości tekstów w zbiorze testowym")
plt.axvline(128, color="red", linestyle="--", label="max_length=128 (orientacyjnie)")
plt.axvline(512, color="green", linestyle="--", label="max_length=512")
plt.legend()
plt.tight_layout()
plt.show()

## Krok 2: Funkcja pomocnicza do eksperymentów

In [ ]:
def run_encoder_experiment(
    model_name,
    max_length=512,
    batch_size=16,
    temperature=1.0,
    model_cache=None,
):
    """Ładuje model, klasyfikuje dane i zwraca metryki, predykcje oraz cache modelu."""
    if model_cache is None:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForSequenceClassification.from_pretrained(model_name)
        device_str = "cuda" if device == 0 else "cpu"
        model.to(device_str)
        model.eval()
        model_cache = (tokenizer, model, device_str)
    else:
        tokenizer, model, device_str = model_cache

    y_pred = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]
        inputs = tokenizer(
            batch,
            truncation=True,
            max_length=max_length,
            padding=True,
            return_tensors="pt",
        ).to(device_str)

        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits / temperature
            probs = F.softmax(logits, dim=-1)
            predictions = torch.argmax(probs, dim=-1)

        for pred_idx in predictions.cpu().tolist():
            label = model.config.id2label[pred_idx]
            mapped = map_text_to_class(label)
            y_pred.append(mapped if mapped else "neutral")

    metrics = evaluate_predictions(y_true, y_pred)
    return metrics, y_pred, model_cache


def results_to_row(eksperyment, wariant, metrics):
    return {
        "eksperyment": eksperyment,
        "wariant": str(wariant),
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_weighted": metrics["f1_weighted"],
    }


def style_results_table(df):
    metric_cols = ["accuracy", "f1_macro", "f1_weighted"]
    return (
        df.style.format({col: "{:.4f}" for col in metric_cols})
        .highlight_max(subset=metric_cols, axis=0, color="#c8e6c9")
        .set_caption("Najlepsza wartość w wierszu podświetlona na zielono")
    )


def plot_experiment_metrics(df, title):
    metric_cols = ["accuracy", "f1_macro", "f1_weighted"]
    labels = {"accuracy": "Accuracy", "f1_macro": "F1 macro", "f1_weighted": "F1 weighted"}

    x = range(len(df))
    width = 0.25
    fig, ax = plt.subplots(figsize=(max(6, len(df) * 1.8), 4))

    for i, col in enumerate(metric_cols):
        ax.bar([pos + i * width for pos in x], df[col], width=width, label=labels[col])

    ax.set_xticks([pos + width for pos in x])
    ax.set_xticklabels(df["wariant"], rotation=15, ha="right")
    ax.set_ylim(0, 1)
    ax.set_ylabel("Wartość metryki")
    ax.set_title(title)
    ax.legend(loc="lower right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

## Eksperyment A: Porównanie modeli

| Model | Opis |
|-------|------|
| `Voicelab/herbert-base-cased-sentiment` | HerBERT fine-tuned na sentyment ogólny (recenzje) |
| `bardsai/finance-sentiment-pl-base` | HerBERT fine-tuned na sentyment finansowy (inna domena) |

In [ ]:
MODELS_TO_COMPARE = [
    "Voicelab/herbert-base-cased-sentiment",
    "bardsai/finance-sentiment-pl-base",
]

model_results = []
for model_name in MODELS_TO_COMPARE:
    print(f"\n>>> Uruchamiam: {model_name}")
    try:
        metrics, _, _ = run_encoder_experiment(model_name, max_length=512)
        print_evaluation(metrics, title=model_name)
        model_results.append(results_to_row("model", model_name.split("/")[-1], metrics))
    except Exception as e:
        print(f"Błąd dla {model_name}: {e}")
        print("Sprawdź nazwę modelu na https://huggingface.co/models?pipeline_tag=text-classification&language=pl")

model_df = pd.DataFrame(model_results)
display(style_results_table(model_df))
plot_experiment_metrics(model_df, "Eksperyment A — porównanie modeli")

## Eksperyment B: Wpływ max_length

Sprawdzamy, czy skracanie kontekstu (128 vs 512 tokenów) zmienia jakość klasyfikacji.

In [ ]:
BASE_MODEL = "Voicelab/herbert-base-cased-sentiment"
MAX_LENGTHS = [128, 256, 512]

length_results = []
y_pred_baseline = None
for max_len in MAX_LENGTHS:
    print(f"\n>>> max_length = {max_len}")
    metrics, y_pred, _ = run_encoder_experiment(BASE_MODEL, max_length=max_len)
    if max_len == 512:
        y_pred_baseline = y_pred
    print_evaluation(metrics, title=f"max_length={max_len}")
    length_results.append(results_to_row("max_length", max_len, metrics))

length_df = pd.DataFrame(length_results)
display(style_results_table(length_df))
plot_experiment_metrics(length_df, "Eksperyment B — wpływ max_length")

## Eksperyment C: Wpływ temperatury

Skalujemy logity przed softmax: `logits / temperature`. Model ładujemy **raz** — przy kolejnych wartościach zmieniamy tylko parametr `temperature` przy inferencji.

In [ ]:
TEMPERATURES = [0.5, 1.0, 2.0]

temperature_results = []
temp_predictions = {}
model_cache = None

for temp in TEMPERATURES:
    print(f"\n>>> temperature = {temp}")
    metrics, y_pred, model_cache = run_encoder_experiment(
        BASE_MODEL,
        max_length=512,
        temperature=temp,
        model_cache=model_cache,
    )
    temp_predictions[temp] = y_pred
    print_evaluation(metrics, title=f"temperature={temp}")
    temperature_results.append(results_to_row("temperature", temp, metrics))

temperature_df = pd.DataFrame(temperature_results)
display(style_results_table(temperature_df))
plot_experiment_metrics(temperature_df, "Eksperyment C — wpływ temperatury")

ref_temp = 1.0
ref_preds = temp_predictions[ref_temp]
print(f"\nZmiany predykcji względem T={ref_temp}:")
for temp in TEMPERATURES:
    changed = sum(p != r for p, r in zip(temp_predictions[temp], ref_preds))
    print(f"  T={temp}: {changed} zmienionych ({100 * changed / len(ref_preds):.1f}%)")

print(
    "\nUwaga: przy argmax skalowanie logitów nie zmienia kolejności klas — "
    "metryki powinny być identyczne. Temperatura wpływa na rozkład prawdopodobieństw, nie na twardą decyzję."
)

## Analiza trudnych przypadków

Sprawdzamy, gdzie model (HerBERT, `max_length=512`) się myli — to pomaga zrozumieć ograniczenia klasyfikatora.

In [ ]:
df_results = pd.DataFrame({"text": sentences, "true": y_true, "pred": y_pred_baseline})
errors = df_results[df_results["true"] != df_results["pred"]]

print(f"Liczba błędów: {len(errors)} / {len(df_results)} ({100 * len(errors) / len(df_results):.1f}%)")
print(f"\nRozkład błędów (true → pred):")
print(errors.groupby(["true", "pred"]).size().sort_values(ascending=False).head(10))

print("\nPrzykłady błędów:")
for _, row in errors.head(5).iterrows():
    print(f"\n  Prawda: {row['true']} | Predykcja: {row['pred']}")
    print(f"  Tekst: {row['text'][:200]}{'...' if len(row['text']) > 200 else ''}")

## Krok 3: Tabela porównawcza i wykres zbiorczy

Wszystkie eksperymenty (A: model, B: max_length, C: temperatura) w jednym widoku — tabele ze stylowaniem i wykres słupkowy.

In [ ]:
comparison_df = pd.DataFrame(model_results + length_results + temperature_results)

EXPERIMENT_LABELS = {
    "model": "A — model",
    "max_length": "B — max_length",
    "temperature": "C — temperatura",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
metric_cols = ["accuracy", "f1_macro", "f1_weighted"]
metric_labels = ["Accuracy", "F1 macro", "F1 weighted"]

for ax, (exp_name, group) in zip(axes, comparison_df.groupby("eksperyment", sort=False)):
    x = range(len(group))
    width = 0.25
    for i, col in enumerate(metric_cols):
        ax.bar([pos + i * width for pos in x], group[col], width=width, label=metric_labels[i])
    ax.set_xticks([pos + width for pos in x])
    ax.set_xticklabels(group["wariant"], rotation=15, ha="right")
    ax.set_ylim(0, 1)
    ax.set_title(EXPERIMENT_LABELS.get(exp_name, exp_name))
    ax.grid(axis="y", alpha=0.3)

axes[0].legend(loc="lower right")
plt.suptitle("Porównanie wszystkich eksperymentów", y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

print("--- Tabele per eksperyment ---")
for exp_name, group in comparison_df.groupby("eksperyment", sort=False):
    print(f"\n{EXPERIMENT_LABELS.get(exp_name, exp_name)}")
    display(
        style_results_table(
            group[["wariant", "accuracy", "f1_macro", "f1_weighted"]].reset_index(drop=True)
        )
    )

comparison_df

## Podsumowanie (do raportu)

- **Analiza długości** — rozkład słów w recenzjach uzasadnia testowanie `max_length` (np. 128 vs 512), zamiast arbitralnego doboru.
- **Model z tej samej domeny** (recenzje) zwykle radzi sobie lepiej niż model z innej domeny (np. finanse).
- **max_length** — zbyt krótki kontekst może obcinać istotne fragmenty długich recenzji; optymalna wartość zależy od metryki (accuracy vs F1 macro).
- **Temperatura** — przy klasyfikacji argmax wyniki są identyczne dla każdej T > 0 (skalowanie logitów nie zmienia wyboru klasy); parametr ma sens przy analizie pewności modelu lub próbkowaniu.
- **Trudne przypadki** — błędy często dotyczą recenzji neutralnych lub mieszanych (np. pozytywny opis z negatywnym wnioskiem).
- Klasa **neutral** jest najtrudniejsza dla obu modeli (niski recall).
- Inne modele PL: [Hugging Face — text-classification (pl)](https://huggingface.co/models?pipeline_tag=text-classification&language=pl).